# Notebook 01 — Crop Registry and Data Loader

**Frozen scope from the governing plan**
- crop registry creation and inspection
- image loading and coordinate validation

This notebook is self-contained with respect to code generation: it bootstraps the `src/spotmeta/` modules it needs into the repo, then imports them through an explicit API sync check. It does **not** perform candidate generation, truth matching, model training, or inference.

**Project-specific interpretation**
- the repo is self-contained except for the data
- initial crop selection is **image-driven and regime-balanced**
- disagreement-focused crops are intentionally deferred until after Notebook 03 exists


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import os
import platform
import re
import sys
import textwrap
import time
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [2]:
# -------------------------------------------------------------------
# REPO DISCOVERY
# -------------------------------------------------------------------

def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / ".git").exists() or (p / "registries").exists():
            return p
    raise RuntimeError("Could not locate repo root from current working directory.")

REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / "src"
NOTEBOOK_NAME = "01_crop_registry_and_data_loader.ipynb"

print("REPO_ROOT =", REPO_ROOT)
print("SRC_ROOT  =", SRC_ROOT)


REPO_ROOT = C:\Users\bpa12\Desktop\spot-detection-meta
SRC_ROOT  = C:\Users\bpa12\Desktop\spot-detection-meta\src


In [7]:
# -------------------------------------------------------------------
# NOTEBOOK 01 CONFIG
# -------------------------------------------------------------------

CFG = {
    "data_root_override": None,
    "data_root_candidates": [
        str(REPO_ROOT / "data"),
        str(Path.home() / "Desktop" / "bpa1"),
        str(Path.home() / "Desktop" / "BPA1"),
        str(Path.home() / "Desktop"),
    ],
    "well_allowlist": ["C8", "D8"],
    "dataset_id": "dataset_default",
    "image_suffixes": [".tif", ".tiff"],
    "recursive_search": True,
    "crop_size_yx": [1024, 1024],
    "max_crops_per_well": 18,
    "edge_margin_px": 24,
    "allow_overlap": False,
    "min_center_separation_px": 384,
    "manifest_subdir": "artifacts/manifests",
    "gallery_subdir": "artifacts/galleries",
    "crop_registry_subdir": "annotations/crop_registry",
    "tables_subdir": "tables",
    "crop_registry_version": "v1_draft",
    "annotator_status_default": "pending",
    "write_yaml_also": True,
    "write_schema_contract_doc": True,
}
CFG


{'data_root_override': None,
 'data_root_candidates': ['C:\\Users\\bpa12\\Desktop\\spot-detection-meta\\data',
  'C:\\Users\\bpa12\\Desktop\\bpa1',
  'C:\\Users\\bpa12\\Desktop\\BPA1',
  'C:\\Users\\bpa12\\Desktop'],
 'well_allowlist': ['C8', 'D8'],
 'dataset_id': 'dataset_default',
 'image_suffixes': ['.tif', '.tiff'],
 'recursive_search': True,
 'crop_size_yx': [256, 256],
 'max_crops_per_well': 18,
 'edge_margin_px': 24,
 'allow_overlap': False,
 'min_center_separation_px': 96,
 'manifest_subdir': 'artifacts/manifests',
 'gallery_subdir': 'artifacts/galleries',
 'crop_registry_subdir': 'annotations/crop_registry',
 'tables_subdir': 'tables',
 'crop_registry_version': 'v1_draft',
 'annotator_status_default': 'needs_annotation',
 'write_yaml_also': True,
 'write_schema_contract_doc': True}

In [ ]:
# -------------------------------------------------------------------
# BOOTSTRAP src/spotmeta MODULES
# -------------------------------------------------------------------

BOOTSTRAP_FILES = {}

BOOTSTRAP_FILES["src/spotmeta/core/__init__.py"] = r"""
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import getpass
import json
import os
import platform
import uuid
from typing import Any

def ensure_dir(path: str | Path) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p

def make_run_id(prefix: str) -> str:
    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    return f"{prefix}_{ts}_{uuid.uuid4().hex[:8]}"

def build_provenance_record(
    *,
    run_id: str,
    notebook_name: str,
    repo_root: str,
    config: dict[str, Any],
) -> dict[str, Any]:
    return {
        "run_id": run_id,
        "notebook_name": notebook_name,
        "repo_root": repo_root,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "user": getpass.getuser(),
        "host": platform.node(),
        "platform": platform.platform(),
        "python_executable": os.sys.executable,
        "config": config,
    }

def write_json(obj: Any, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, sort_keys=False), encoding="utf-8")
    return path
"""

BOOTSTRAP_FILES["src/spotmeta/io/__init__.py"] = r"""
from .discovery import (
    choose_discovery_root,
    inventory_image_files,
    enrich_inventory_with_shapes,
    load_image_array,
    compute_projection,
)
"""

BOOTSTRAP_FILES["src/spotmeta/io/discovery.py"] = r"""
from __future__ import annotations

from pathlib import Path
import re
from typing import Iterable

import numpy as np
import pandas as pd
import tifffile

WELL_PAT = re.compile(r"(?<![A-Z0-9])([A-H](?:[1-9]|1[0-2]))(?![A-Z0-9])", re.IGNORECASE)
CYCLE_PAT = re.compile(r"(?:cycle|round|r)(?:[_\-\s]?)(\d+)", re.IGNORECASE)
CHANNEL_PAT = re.compile(r"(?:channel|chan|ch|c)(?:[_\-\s]?)(\d+)", re.IGNORECASE)

def choose_discovery_root(candidates: list[str], override: str | None = None) -> Path:
    if override:
        p = Path(override).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(f"Configured data_root_override does not exist: {p}")
        return p

    existing = []
    for cand in candidates:
        p = Path(cand).expanduser().resolve()
        if p.exists():
            n_img = sum(1 for x in p.rglob("*") if x.is_file() and x.suffix.lower() in {".tif", ".tiff"})
            existing.append((n_img, p))
    if not existing:
        raise FileNotFoundError("None of the candidate data roots exists.")
    existing.sort(key=lambda t: (t[0], len(str(t[1]))), reverse=True)
    return existing[0][1]

def _extract_well(path: Path) -> str | None:
    joined = " ".join(path.parts[-4:]) + " " + path.name
    m = WELL_PAT.search(joined)
    return m.group(1).upper() if m else None

def _extract_cycle(path: Path) -> int | None:
    joined = " ".join(path.parts[-4:]) + " " + path.name
    m = CYCLE_PAT.search(joined)
    return int(m.group(1)) if m else None

def _extract_channel(path: Path) -> int | None:
    joined = " ".join(path.parts[-4:]) + " " + path.name
    m = CHANNEL_PAT.search(joined)
    return int(m.group(1)) if m else None

def inventory_image_files(
    root: str | Path,
    *,
    suffixes: Iterable[str] = (".tif", ".tiff"),
    recursive: bool = True,
    dataset_id: str = "dataset_default",
) -> pd.DataFrame:
    root = Path(root).expanduser().resolve()
    suffixes = {s.lower() for s in suffixes}
    iterator = root.rglob("*") if recursive else root.glob("*")
    rows = []
    for path in iterator:
        if not path.is_file():
            continue
        if path.suffix.lower() not in suffixes:
            continue
        rows.append({
            "dataset_id": dataset_id,
            "file_path": str(path),
            "file_name": path.name,
            "suffix": path.suffix.lower(),
            "well_id": _extract_well(path),
            "cycle_id": _extract_cycle(path),
            "channel_id": _extract_channel(path),
        })
    df = pd.DataFrame(rows)
    if len(df) == 0:
        return pd.DataFrame(columns=["dataset_id", "file_path", "file_name", "suffix", "well_id", "cycle_id", "channel_id"])
    return df.sort_values(["well_id", "cycle_id", "channel_id", "file_name"], na_position="last").reset_index(drop=True)

def load_image_array(path: str | Path) -> np.ndarray:
    arr = tifffile.imread(str(path))
    arr = np.asarray(arr)
    return arr

def _is_valid_tiff(path: str | Path) -> bool:
    try:
        import tifffile as _tf
        with _tf.TiffFile(str(path)):
            pass
        return True
    except Exception:
        return False

def _infer_yx(arr: np.ndarray) -> tuple[int, int]:
    if arr.ndim < 2:
        raise ValueError(f"Expected image with ndim>=2, got shape {arr.shape}")
    return int(arr.shape[-2]), int(arr.shape[-1])

def enrich_inventory_with_shapes(df: pd.DataFrame) -> pd.DataFrame:
    if len(df) == 0:
        out = df.copy()
        out["image_shape_y"] = pd.Series(dtype="Int64")
        out["image_shape_x"] = pd.Series(dtype="Int64")
        out["image_ndim"] = pd.Series(dtype="Int64")
        out["image_dtype"] = pd.Series(dtype="object")
        return out

    rows = []
    skipped = []
    for rec in df.to_dict(orient="records"):
        try:
            arr = load_image_array(rec["file_path"])
            y, x = _infer_yx(arr)
            rec = dict(rec)
            rec["image_shape_y"] = y
            rec["image_shape_x"] = x
            rec["image_ndim"] = int(arr.ndim)
            rec["image_dtype"] = str(arr.dtype)
            rows.append(rec)
        except Exception as _e:
            skipped.append((rec["file_path"], str(_e)))
    if skipped:
        import warnings
        for _path, _reason in skipped:
            warnings.warn(f"Skipped non-readable file: {_path}\n  reason: {_reason}")
    return pd.DataFrame(rows)

def compute_projection(arr: np.ndarray, kind: str = "max") -> np.ndarray:
    arr = np.asarray(arr)
    if arr.ndim == 2:
        proj = arr
    elif arr.ndim >= 3:
        axes = tuple(range(arr.ndim - 2))
        if kind == "max":
            proj = np.max(arr, axis=axes)
        elif kind == "mean":
            proj = np.mean(arr, axis=axes)
        else:
            raise ValueError(f"Unsupported projection kind: {kind}")
    else:
        raise ValueError(f"Unsupported image shape: {arr.shape}")
    return np.asarray(proj, dtype=np.float32)
"""

BOOTSTRAP_FILES["src/spotmeta/truth/__init__.py"] = r"""
from .crop_registry import (
    summarize_inventory,
    select_primary_well_images,
    propose_crops_for_projection,
    deduplicate_crop_records,
    build_crop_registry,
    append_manual_crops,
    write_crop_registry_yaml,
    read_crop_registry_yaml,
    write_schema_contract_doc,
)
"""

BOOTSTRAP_FILES["src/spotmeta/truth/crop_registry.py"] = r"""
from __future__ import annotations

from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
import yaml

def summarize_inventory(df: pd.DataFrame) -> pd.DataFrame:
    if len(df) == 0:
        return pd.DataFrame(columns=["well_id", "n_files", "n_cycles", "n_channels", "image_shape_y", "image_shape_x"])
    grp = (
        df.groupby("well_id", dropna=False)
        .agg(
            n_files=("file_path", "size"),
            n_cycles=("cycle_id", lambda s: int(pd.Series(s).dropna().nunique())),
            n_channels=("channel_id", lambda s: int(pd.Series(s).dropna().nunique())),
            image_shape_y=("image_shape_y", "first"),
            image_shape_x=("image_shape_x", "first"),
        )
        .reset_index()
        .sort_values("well_id", na_position="last")
        .reset_index(drop=True)
    )
    return grp

def select_primary_well_images(df: pd.DataFrame) -> pd.DataFrame:
    if len(df) == 0:
        return df.copy()
    work = df.copy()
    work["cycle_rank"] = work["cycle_id"].fillna(10**9)
    work["channel_rank"] = work["channel_id"].fillna(10**9)
    work = work.sort_values(["well_id", "cycle_rank", "channel_rank", "file_name"], na_position="last")
    primary = work.groupby("well_id", as_index=False).first()
    return primary.drop(columns=["cycle_rank", "channel_rank"])

def _local_stats(img: np.ndarray, y0: int, x0: int, h: int, w: int) -> dict:
    patch = img[y0:y0+h, x0:x0+w]
    if patch.size == 0:
        return {"mean": -np.inf, "std": -np.inf, "max": -np.inf, "q99": -np.inf, "edge": -np.inf, "closepair": -np.inf}
    gy, gx = np.gradient(patch.astype(np.float32), axis=(0, 1))
    edge = float(np.mean(np.hypot(gy, gx)))
    mx = float(np.max(patch))
    mean = float(np.mean(patch))
    std = float(np.std(patch))
    q99 = float(np.quantile(patch, 0.99))
    bright = patch >= np.quantile(patch, 0.995)
    closepair = float(np.sum(bright))
    return {"mean": mean, "std": std, "max": mx, "q99": q99, "edge": edge, "closepair": closepair}

def _candidate_grid(shape_y: int, shape_x: int, crop_h: int, crop_w: int, step: int):
    y_limit = max(1, shape_y - crop_h + 1)
    x_limit = max(1, shape_x - crop_w + 1)
    for y0 in range(0, y_limit, max(1, step)):
        for x0 in range(0, x_limit, max(1, step)):
            yield y0, x0
    if (shape_y - crop_h) not in range(0, y_limit, max(1, step)) or (shape_x - crop_w) not in range(0, x_limit, max(1, step)):
        yield max(0, shape_y - crop_h), max(0, shape_x - crop_w)

def _make_crop_id(dataset_id: str, well_id: str, y0: int, x0: int, y1: int, x1: int) -> str:
    raw = f"{dataset_id}|{well_id}|{y0}|{x0}|{y1}|{x1}"
    digest = hashlib.sha1(raw.encode("utf-8")).hexdigest()[:12]
    return f"crop_{well_id}_{digest}"

def propose_crops_for_projection(
    projection: np.ndarray,
    *,
    dataset_id: str,
    well_id: str,
    crop_size_yx: tuple[int, int] = (256, 256),
    max_crops: int = 12,
    edge_margin_px: int = 24,
    min_center_separation_px: int = 128,
    allow_overlap: bool = False,
) -> list[dict]:
    img = np.asarray(projection, dtype=np.float32)
    H, W = img.shape
    crop_h, crop_w = map(int, crop_size_yx)
    crop_h = min(crop_h, H)
    crop_w = min(crop_w, W)
    step = max(32, min(crop_h, crop_w) // 2)

    stats = []
    for y0, x0 in _candidate_grid(H, W, crop_h, crop_w, step):
        y1, x1 = y0 + crop_h, x0 + crop_w
        s = _local_stats(img, y0, x0, crop_h, crop_w)
        s.update({"y0": y0, "x0": x0, "y1": y1, "x1": x1})
        s["is_edge"] = int(y0 <= edge_margin_px or x0 <= edge_margin_px or y1 >= H - edge_margin_px or x1 >= W - edge_margin_px)
        stats.append(s)

    if not stats:
        return []

    sdf = pd.DataFrame(stats)
    pools = {
        "bright": sdf.sort_values(["q99", "max"], ascending=False),
        "dim": sdf.sort_values(["mean", "q99"], ascending=True),
        "dense": sdf.sort_values(["std", "q99"], ascending=False),
        "sparse": sdf.sort_values(["std", "mean"], ascending=True),
        "close_pair": sdf.sort_values(["closepair", "std"], ascending=False),
        "edge_artifact": sdf.sort_values(["is_edge", "edge", "q99"], ascending=False),
    }

    wanted_tags = ["bright", "dim", "dense", "sparse", "close_pair", "edge_artifact"]
    chosen = []

    def center(rec):
        return ((rec["y0"] + rec["y1"]) / 2.0, (rec["x0"] + rec["x1"]) / 2.0)

    def too_close(rec):
        cy, cx = center(rec)
        for prev in chosen:
            py, px = center(prev)
            if np.hypot(cy - py, cx - px) < min_center_separation_px:
                return True
            if not allow_overlap:
                if not (rec["x1"] <= prev["x0"] or rec["x0"] >= prev["x1"] or rec["y1"] <= prev["y0"] or rec["y0"] >= prev["y1"]):
                    return True
        return False

    for tag in wanted_tags:
        for _, rec in pools[tag].iterrows():
            rec = rec.to_dict()
            if too_close(rec):
                continue
            chosen.append(rec | {"primary_tag": tag})
            break

    sdf["composite"] = (
        sdf["q99"].rank(pct=True)
        + sdf["std"].rank(pct=True)
        + sdf["edge"].rank(pct=True)
        + sdf["closepair"].rank(pct=True)
    )
    for _, rec in sdf.sort_values("composite", ascending=False).iterrows():
        if len(chosen) >= max_crops:
            break
        rec = rec.to_dict()
        if too_close(rec):
            continue
        chosen.append(rec | {"primary_tag": "mixed"})

    records = []
    for rec in chosen[:max_crops]:
        y0, x0, y1, x1 = map(int, [rec["y0"], rec["x0"], rec["y1"], rec["x1"]])
        tags = sorted(set([rec["primary_tag"]]))
        if rec.get("is_edge", 0):
            tags.append("edge_artifact")
        if rec["q99"] >= sdf["q99"].quantile(0.8):
            tags.append("bright")
        if rec["mean"] <= sdf["mean"].quantile(0.2):
            tags.append("dim")
        if rec["std"] >= sdf["std"].quantile(0.8):
            tags.append("dense")
        if rec["std"] <= sdf["std"].quantile(0.2):
            tags.append("sparse")
        if rec["closepair"] >= sdf["closepair"].quantile(0.8):
            tags.append("close_pair")
        tags = sorted(set(tags))

        rationale = (
            f"image-driven initial crop; primary_tag={rec['primary_tag']}; "
            f"mean={rec['mean']:.3f}; std={rec['std']:.3f}; q99={rec['q99']:.3f}; edge={rec['edge']:.3f}; closepair={rec['closepair']:.1f}"
        )
        records.append({
            "crop_id": _make_crop_id(dataset_id, well_id, y0, x0, y1, x1),
            "dataset_id": dataset_id,
            "well_id": well_id,
            "well_ymin_px": y0,
            "well_xmin_px": x0,
            "well_ymax_px": y1,
            "well_xmax_px": x1,
            "selection_tags": tags,
            "selection_rationale": rationale,
        })
    return records

def deduplicate_crop_records(records: list[dict]) -> list[dict]:
    out = []
    seen = set()
    for rec in records:
        cid = rec["crop_id"]
        if cid in seen:
            continue
        seen.add(cid)
        out.append(rec)
    return out

def build_crop_registry(
    records: list[dict],
    *,
    crop_registry_version: str,
    annotator_status_default: str = "pending",
) -> pd.DataFrame:
    rows = []
    for rec in deduplicate_crop_records(records):
        row = dict(rec)
        row["annotator_status"] = row.get("annotator_status", annotator_status_default)
        row["crop_registry_version"] = row.get("crop_registry_version", crop_registry_version)
        rows.append(row)
    cols = [
        "crop_id", "dataset_id", "well_id",
        "well_ymin_px", "well_xmin_px", "well_ymax_px", "well_xmax_px",
        "selection_tags", "selection_rationale", "annotator_status", "crop_registry_version",
    ]
    out = pd.DataFrame(rows)
    if len(out) == 0:
        return pd.DataFrame(columns=cols)
    return out[cols].sort_values(["well_id", "well_ymin_px", "well_xmin_px"]).reset_index(drop=True)

def append_manual_crops(df: pd.DataFrame, manual_records: list[dict], *, crop_registry_version: str) -> pd.DataFrame:
    if not manual_records:
        return df.copy()
    extra = build_crop_registry(
        manual_records,
        crop_registry_version=crop_registry_version,
        annotator_status_default="pending",
    )
    merged = pd.concat([df, extra], ignore_index=True)
    merged = merged.drop_duplicates(subset=["crop_id"], keep="first").reset_index(drop=True)
    return merged

def crop_registry_to_yaml_records(df: pd.DataFrame) -> list[dict]:
    out = []
    for rec in df.to_dict(orient="records"):
        rec = dict(rec)
        rec["selection_tags"] = list(rec["selection_tags"])
        out.append(rec)
    return out

def write_crop_registry_yaml(df: pd.DataFrame, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "format": "spotmeta_crop_registry",
        "n_rows": int(len(df)),
        "records": crop_registry_to_yaml_records(df),
    }
    path.write_text(yaml.safe_dump(payload, sort_keys=False), encoding="utf-8")
    return path

def read_crop_registry_yaml(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    payload = yaml.safe_load(path.read_text(encoding="utf-8"))
    return pd.DataFrame(payload["records"])

def write_schema_contract_doc(path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    lines = [
        "# Crop registry schema contract",
        "",
        "Required fields:",
        "- crop_id",
        "- dataset_id",
        "- well_id",
        "- well_ymin_px",
        "- well_xmin_px",
        "- well_ymax_px",
        "- well_xmax_px",
        "- selection_tags",
        "- selection_rationale",
        "- annotator_status",
        "- crop_registry_version",
        "",
        "Coordinate contract:",
        "- bounds are in full-well pixel frame (well_y_px, well_x_px)",
        "- ymin/xmin inclusive, ymax/xmax exclusive",
        "- coord_frame_primary: well",
        "- coord_units: pixel",
        "- crop_origin_well_y_px = well_ymin_px",
        "- crop_origin_well_x_px = well_xmin_px",
    ]
    text = "\n".join(lines) + "\n"
    path.write_text(text, encoding="utf-8")
    return path
"""

BOOTSTRAP_FILES["src/spotmeta/validation/__init__.py"] = r"""
from .crop_validation import (
    validate_crop_registry,
    assert_roundtrip_examples,
    normalize_registry_dataframe,
)
"""

BOOTSTRAP_FILES["src/spotmeta/validation/crop_validation.py"] = r"""
from __future__ import annotations

import pandas as pd

REQUIRED_CROP_COLS = [
    "crop_id", "dataset_id", "well_id",
    "well_ymin_px", "well_xmin_px", "well_ymax_px", "well_xmax_px",
    "selection_tags", "selection_rationale", "annotator_status", "crop_registry_version",
]
ALLOWED_STATUS = {"pending", "in_progress", "complete", "excluded"}

def normalize_registry_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "selection_tags" in out.columns:
        out["selection_tags"] = out["selection_tags"].apply(
            lambda x: list(x) if isinstance(x, (list, tuple, set)) else ([] if pd.isna(x) else [str(x)])
        )
    for c in ["well_ymin_px", "well_xmin_px", "well_ymax_px", "well_xmax_px"]:
        if c in out.columns:
            out[c] = out[c].astype(int)
    return out

def validate_crop_registry(df: pd.DataFrame) -> None:
    missing = [c for c in REQUIRED_CROP_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Crop registry missing required columns: {missing}")
    d = normalize_registry_dataframe(df)
    if d["crop_id"].duplicated().any():
        raise ValueError("crop_id values must be unique")
    bad = d[~d["annotator_status"].isin(ALLOWED_STATUS)]
    if len(bad):
        raise ValueError(f"Invalid annotator_status values: {sorted(bad['annotator_status'].unique())}")
    for _, r in d.iterrows():
        if not (r["well_ymin_px"] < r["well_ymax_px"] and r["well_xmin_px"] < r["well_xmax_px"]):
            raise ValueError(f"Invalid crop bounds for crop_id={r['crop_id']}")
        if not isinstance(r["selection_tags"], list):
            raise ValueError(f"selection_tags must be list for crop_id={r['crop_id']}")

def assert_roundtrip_examples(df: pd.DataFrame, shape_lookup: dict[tuple[str, str], tuple[int, int]]) -> None:
    d = normalize_registry_dataframe(df)
    for _, r in d.iterrows():
        key = (r["dataset_id"], r["well_id"])
        if key not in shape_lookup:
            continue
        H, W = shape_lookup[key]
        if not (0 <= r["well_ymin_px"] < r["well_ymax_px"] <= H):
            raise ValueError(f"Crop y-bounds exceed image shape for {r['crop_id']}: {(r['well_ymin_px'], r['well_ymax_px'])} vs H={H}")
        if not (0 <= r["well_xmin_px"] < r["well_xmax_px"] <= W):
            raise ValueError(f"Crop x-bounds exceed image shape for {r['crop_id']}: {(r['well_xmin_px'], r['well_xmax_px'])} vs W={W}")
"""

BOOTSTRAP_FILES["src/spotmeta/viz/__init__.py"] = r"""
from .crop_viz import plot_crop_overlay, plot_crop_gallery
"""

BOOTSTRAP_FILES["src/spotmeta/viz/crop_viz.py"] = r"""
from __future__ import annotations

from pathlib import Path
import math

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd

def plot_crop_overlay(
    image: np.ndarray,
    crop_df: pd.DataFrame,
    *,
    title: str = "",
    save_path: str | Path | None = None,
):
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image, cmap="gray")
    for _, r in crop_df.iterrows():
        rect = Rectangle(
            (r["well_xmin_px"], r["well_ymin_px"]),
            r["well_xmax_px"] - r["well_xmin_px"],
            r["well_ymax_px"] - r["well_ymin_px"],
            fill=False,
            linewidth=1.5,
        )
        ax.add_patch(rect)
        ax.text(r["well_xmin_px"], r["well_ymin_px"], r["crop_id"][-6:], fontsize=6)
    ax.set_title(title)
    ax.set_axis_off()
    fig.tight_layout()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return fig, ax

def plot_crop_gallery(
    image: np.ndarray,
    crop_df: pd.DataFrame,
    *,
    title: str = "",
    save_path: str | Path | None = None,
):
    n = len(crop_df)
    ncols = 3
    nrows = max(1, math.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.set_axis_off()
    for ax, (_, r) in zip(axes, crop_df.iterrows()):
        patch = image[r["well_ymin_px"]:r["well_ymax_px"], r["well_xmin_px"]:r["well_xmax_px"]]
        ax.imshow(patch, cmap="gray")
        ax.set_title(f"{r['well_id']} | {','.join(r['selection_tags'])}")
        ax.set_axis_off()
    fig.suptitle(title)
    fig.tight_layout()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return fig, axes
"""

SyntaxError: invalid syntax (1366346103.py, line 457)

In [12]:
# -------------------------------------------------------------------
# MATERIALIZE BOOTSTRAP FILES
# -------------------------------------------------------------------

written_files = []
for rel_path, text in BOOTSTRAP_FILES.items():
    path = REPO_ROOT / rel_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(text).lstrip("\n"), encoding="utf-8", newline="\n")
    written_files.append(path)

print(f"Wrote {len(written_files)} bootstrap files.")
for p in written_files:
    print(" -", p.relative_to(REPO_ROOT))


Wrote 9 bootstrap files.
 - src\spotmeta\core\__init__.py
 - src\spotmeta\io\__init__.py
 - src\spotmeta\io\discovery.py
 - src\spotmeta\truth\__init__.py
 - src\spotmeta\truth\crop_registry.py
 - src\spotmeta\validation\__init__.py
 - src\spotmeta\validation\crop_validation.py
 - src\spotmeta\viz\__init__.py
 - src\spotmeta\viz\crop_viz.py


In [13]:
# -------------------------------------------------------------------
# API SYNC / IMPORT CHECK
# -------------------------------------------------------------------

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import importlib

for name in list(sys.modules):
    if name == "spotmeta" or name.startswith("spotmeta."):
        del sys.modules[name]
importlib.invalidate_caches()

NOTEBOOK01_API = {
    "spotmeta.core": ["make_run_id", "ensure_dir", "build_provenance_record", "write_json"],
    "spotmeta.io": ["choose_discovery_root", "inventory_image_files", "enrich_inventory_with_shapes", "load_image_array", "compute_projection"],
    "spotmeta.truth": ["summarize_inventory", "select_primary_well_images", "propose_crops_for_projection", "deduplicate_crop_records", "build_crop_registry", "append_manual_crops", "write_crop_registry_yaml", "read_crop_registry_yaml", "write_schema_contract_doc"],
    "spotmeta.validation": ["validate_crop_registry", "assert_roundtrip_examples", "normalize_registry_dataframe"],
    "spotmeta.viz": ["plot_crop_overlay", "plot_crop_gallery"],
}

missing = []
for module_name, symbols in NOTEBOOK01_API.items():
    mod = importlib.import_module(module_name)
    for sym in symbols:
        if hasattr(mod, sym):
            globals()[sym] = getattr(mod, sym)
        else:
            missing.append(f"{module_name}.{sym}")

if missing:
    raise RuntimeError("Missing notebook API symbols:\n" + "\n".join(missing))

print("Notebook 01 API sync succeeded.")


SyntaxError: unexpected character after line continuation character (crop_registry.py, line 251)

In [ ]:
# -------------------------------------------------------------------
# RUN METADATA
# -------------------------------------------------------------------

RUN_ID = make_run_id("nb01")
RUN_START_TS = time.time()

MANIFEST_DIR = ensure_dir(REPO_ROOT / CFG["manifest_subdir"])
GALLERY_DIR = ensure_dir(REPO_ROOT / CFG["gallery_subdir"])
CROP_REGISTRY_DIR = ensure_dir(REPO_ROOT / CFG["crop_registry_subdir"])
TABLES_DIR = ensure_dir(REPO_ROOT / CFG["tables_subdir"])
DOC_SPECS_DIR = ensure_dir(REPO_ROOT / "docs" / "specs")

RUN_PROVENANCE = build_provenance_record(
    run_id=RUN_ID,
    notebook_name=NOTEBOOK_NAME,
    repo_root=str(REPO_ROOT),
    config=CFG,
)

RUN_PROVENANCE


In [ ]:
# -------------------------------------------------------------------
# DISCOVER DATA ROOT
# -------------------------------------------------------------------

DATA_ROOT = choose_discovery_root(
    candidates=CFG["data_root_candidates"],
    override=CFG["data_root_override"],
)
print("Selected DATA_ROOT:")
print(DATA_ROOT)


In [ ]:
# -------------------------------------------------------------------
# INVENTORY IMAGE FILES
# -------------------------------------------------------------------

inventory_df = inventory_image_files(
    DATA_ROOT,
    suffixes=CFG["image_suffixes"],
    recursive=CFG["recursive_search"],
    dataset_id=CFG["dataset_id"],
)

print("Discovered image files:", len(inventory_df))
display(inventory_df.head(20))


In [ ]:
# -------------------------------------------------------------------
# OPTIONAL WELL FILTER
# -------------------------------------------------------------------

allow = [w.upper() for w in CFG.get("well_allowlist", []) if str(w).strip()]
if allow:
    inventory_df = inventory_df[inventory_df["well_id"].isin(allow)].reset_index(drop=True)
    print("Filtered wells:", allow)
    print("Remaining image files:", len(inventory_df))
else:
    print("No well filter applied.")
display(inventory_df.head(20))


In [ ]:
# -------------------------------------------------------------------
# ENRICH INVENTORY WITH SHAPES
# -------------------------------------------------------------------

inventory_df = enrich_inventory_with_shapes(inventory_df)
inventory_summary_df = summarize_inventory(inventory_df)

print("Inventory summary by well:")
display(inventory_summary_df)


In [ ]:
# -------------------------------------------------------------------
# PERSIST INVENTORY TABLES
# -------------------------------------------------------------------

inventory_parquet = TABLES_DIR / f"{RUN_ID}_image_inventory.parquet"
inventory_summary_parquet = TABLES_DIR / f"{RUN_ID}_image_inventory_summary.parquet"

inventory_df.to_parquet(inventory_parquet, index=False)
inventory_summary_df.to_parquet(inventory_summary_parquet, index=False)

print(inventory_parquet)
print(inventory_summary_parquet)


In [ ]:
# -------------------------------------------------------------------
# PRIMARY IMAGE SELECTION PER WELL
# -------------------------------------------------------------------

primary_df = select_primary_well_images(inventory_df)
display(primary_df)


In [ ]:
# -------------------------------------------------------------------
# LOAD PRIMARY PROJECTIONS
# -------------------------------------------------------------------

primary_images = {}
primary_projections = {}
shape_lookup = {}

for rec in primary_df.to_dict(orient="records"):
    arr = load_image_array(rec["file_path"])
    proj = compute_projection(arr, kind="max")
    key = (rec["dataset_id"], rec["well_id"])
    primary_images[key] = arr
    primary_projections[key] = proj
    shape_lookup[key] = tuple(map(int, proj.shape))

print("Loaded primary projections for:")
for key, shape in shape_lookup.items():
    print(" -", key, shape)


In [ ]:
# -------------------------------------------------------------------
# VISUAL AUDIT OF PRIMARY PROJECTIONS
# -------------------------------------------------------------------

n = len(primary_projections)
fig, axes = plt.subplots(1, max(1, n), figsize=(6 * max(1, n), 6))
axes = np.array(axes).reshape(-1)

for ax, ((dataset_id, well_id), proj) in zip(axes, primary_projections.items()):
    ax.imshow(proj, cmap="gray")
    ax.set_title(f"{dataset_id} | {well_id} | {proj.shape}")
    ax.set_axis_off()

for ax in axes[n:]:
    ax.set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
# -------------------------------------------------------------------
# PROPOSE INITIAL CROPS (IMAGE-DRIVEN ONLY)
# -------------------------------------------------------------------

crop_records = []
for _, rec in primary_df.iterrows():
    key = (rec["dataset_id"], rec["well_id"])
    proj = primary_projections[key]
    proposed = propose_crops_for_projection(
        proj,
        dataset_id=rec["dataset_id"],
        well_id=rec["well_id"],
        crop_size_yx=tuple(CFG["crop_size_yx"]),
        max_crops=int(CFG["max_crops_per_well"]),
        edge_margin_px=int(CFG["edge_margin_px"]),
        min_center_separation_px=int(CFG["min_center_separation_px"]),
        allow_overlap=bool(CFG["allow_overlap"]),
    )
    crop_records.extend(proposed)

crop_records = deduplicate_crop_records(crop_records)
crop_registry_df = build_crop_registry(
    crop_records,
    crop_registry_version=CFG["crop_registry_version"],
    annotator_status_default=CFG["annotator_status_default"],
)

print("Proposed crops:", len(crop_registry_df))
display(crop_registry_df)


In [ ]:
# -------------------------------------------------------------------
# OPTIONAL MANUAL CROP APPEND BLOCK
# -------------------------------------------------------------------

MANUAL_CROP_RECORDS = [
    # {
    #     "crop_id": "crop_C8_manual_001",
    #     "dataset_id": CFG["dataset_id"],
    #     "well_id": "C8",
    #     "well_ymin_px": 100,
    #     "well_xmin_px": 200,
    #     "well_ymax_px": 356,
    #     "well_xmax_px": 456,
    #     "selection_tags": ["manual_review"],
    #     "selection_rationale": "analyst-added crop after visual review",
    # }
]

crop_registry_df = append_manual_crops(
    crop_registry_df,
    MANUAL_CROP_RECORDS,
    crop_registry_version=CFG["crop_registry_version"],
)
crop_registry_df = normalize_registry_dataframe(crop_registry_df)
display(crop_registry_df)


In [ ]:
# -------------------------------------------------------------------
# VALIDATE CROP REGISTRY AGAINST SCHEMA AND IMAGE SHAPES
# -------------------------------------------------------------------

validate_crop_registry(crop_registry_df)
assert_roundtrip_examples(crop_registry_df, shape_lookup)

print("Crop registry validation passed.")


In [ ]:
# -------------------------------------------------------------------
# PER-WELL CROP COUNTS
# -------------------------------------------------------------------

per_well_counts = (
    crop_registry_df.groupby("well_id", as_index=False)
    .agg(n_crops=("crop_id", "size"))
    .sort_values("well_id")
    .reset_index(drop=True)
)
display(per_well_counts)


In [ ]:
# -------------------------------------------------------------------
# OVERLAY / GALLERY ARTIFACTS
# -------------------------------------------------------------------

overlay_paths = []
gallery_paths = []

for (dataset_id, well_id), proj in primary_projections.items():
    crop_sub = crop_registry_df[
        (crop_registry_df["dataset_id"] == dataset_id) &
        (crop_registry_df["well_id"] == well_id)
    ].reset_index(drop=True)
    if len(crop_sub) == 0:
        continue

    overlay_path = GALLERY_DIR / f"{RUN_ID}_{well_id}_overlay.png"
    gallery_path = GALLERY_DIR / f"{RUN_ID}_{well_id}_gallery.png"

    plot_crop_overlay(
        proj, crop_sub,
        title=f"{dataset_id} | {well_id} | crop overlay",
        save_path=overlay_path,
    )
    plt.show()

    plot_crop_gallery(
        proj, crop_sub,
        title=f"{dataset_id} | {well_id} | crop gallery",
        save_path=gallery_path,
    )
    plt.show()

    overlay_paths.append(str(overlay_path))
    gallery_paths.append(str(gallery_path))

overlay_paths, gallery_paths


In [ ]:
# -------------------------------------------------------------------
# PERSIST CROP REGISTRY
# -------------------------------------------------------------------

crop_registry_parquet = CROP_REGISTRY_DIR / f"{RUN_ID}_crop_registry.parquet"
crop_registry_df.to_parquet(crop_registry_parquet, index=False)

crop_registry_yaml = None
if CFG.get("write_yaml_also", True):
    crop_registry_yaml = CROP_REGISTRY_DIR / f"{RUN_ID}_crop_registry.yaml"
    write_crop_registry_yaml(crop_registry_df, crop_registry_yaml)
    roundtrip_df = read_crop_registry_yaml(crop_registry_yaml)
    validate_crop_registry(roundtrip_df)

print("Wrote:")
print(" -", crop_registry_parquet)
if crop_registry_yaml is not None:
    print(" -", crop_registry_yaml)


In [ ]:
# -------------------------------------------------------------------
# WRITE SCHEMA DOC + MANIFEST
# -------------------------------------------------------------------

schema_doc_path = None
if CFG.get("write_schema_contract_doc", True):
    schema_doc_path = write_schema_contract_doc(DOC_SPECS_DIR / "crop_registry_schema_contract.md")

elapsed_sec = time.time() - RUN_START_TS
run_manifest = {
    "run_id": RUN_ID,
    "notebook_name": NOTEBOOK_NAME,
    "data_root": str(DATA_ROOT),
    "inventory_rows": int(len(inventory_df)),
    "n_wells": int(inventory_df["well_id"].nunique()) if len(inventory_df) else 0,
    "n_crops": int(len(crop_registry_df)),
    "per_well_crop_counts": per_well_counts.to_dict(orient="records"),
    "crop_registry_parquet": str(crop_registry_parquet),
    "crop_registry_yaml": None if crop_registry_yaml is None else str(crop_registry_yaml),
    "schema_doc_path": None if schema_doc_path is None else str(schema_doc_path),
    "overlay_paths": overlay_paths,
    "gallery_paths": gallery_paths,
    "elapsed_sec": elapsed_sec,
    "provenance": RUN_PROVENANCE,
}

manifest_path = MANIFEST_DIR / f"{RUN_ID}_run_manifest.json"
write_json(run_manifest, manifest_path)

print(manifest_path)
run_manifest


## Exit criteria for Notebook 01

Notebook 01 is complete when:
- image inventory is persisted
- coordinate and shape assumptions have been validated
- an initial image-driven crop registry exists and passes validation
- overlay/gallery artifacts have been generated for review
- the crop registry is ready for Notebook 02 annotation

Later, after Notebook 03 exists, a **new crop registry version** can be created to add detector-disagreement / error-analysis crops without overwriting this initial image-driven baseline.
